# Speech Coach — GPU Backend Server

**Run this notebook on Colab with a T4 GPU runtime.**

It starts a FastAPI server that your local webapp calls for pipeline processing.

### Setup (one-time):
1. Open this notebook in Google Colab
2. Runtime → Change runtime type → **T4 GPU**
3. **Run All** (Ctrl+F9)
4. Copy the ngrok URL printed at the bottom
5. Paste it in your webapp Settings → COLAB_BACKEND_URL

After that, all uploads from `localhost:3000` are processed on the T4 GPU automatically.

In [ ]:
#@title 1. Install Dependencies
# Colab already has PyTorch+CUDA, so don't reinstall it
!pip install -q transformers librosa noisereduce
!pip install -q praat-parselmouth
!pip install -q openai-whisper
!pip install -q opencv-python mediapipe ultralytics
!pip install -q spacy textstat sentence-transformers
!pip install -q speechbrain
!pip install -q language_tool_python
!pip install -q fastapi uvicorn python-multipart pyngrok
!python -m spacy download en_core_web_sm -q
print('\u2705 Dependencies installed')

In [ ]:
#@title 2. Clone / Update Repo
import os

REPO_URL = 'https://github.com/anvay-cpu/voice-analysis-pipeline.git'
REPO_DIR = '/content/voice-analysis-pipeline'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'\u2705 Repo ready at {REPO_DIR}')

In [ ]:
#@title 3. Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print('\u2705 T4 GPU ready')
else:
    print('\u26a0\ufe0f No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
#@title 3b. Fix missing models & paths
#@markdown Downloads missing MediaPipe models and fixes path mismatches.
import os
os.chdir('/content/voice-analysis-pipeline')

# Download pose model if missing
if not os.path.exists('models/pose_landmarker_lite.task'):
    !wget -q -O models/pose_landmarker_lite.task "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
    print(f'\u2705 Pose model downloaded: {os.path.getsize("models/pose_landmarker_lite.task")/1e6:.1f}MB')
else:
    print('\u2705 Pose model: exists')

# Fix filler verifier symlink (code expects models/filler/)
os.makedirs('models/filler', exist_ok=True)
if not os.path.exists('models/filler/best_model.pt'):
    if os.path.exists('models/filler_verifier/best_model.pt'):
        os.symlink('/content/voice-analysis-pipeline/models/filler_verifier/best_model.pt',
                   'models/filler/best_model.pt')
        print('\u2705 Filler verifier: symlinked')
    else:
        print('\u26a0\ufe0f Filler verifier: not found (regex fallback will be used)')
else:
    print('\u2705 Filler verifier: exists')

# Create disfluency dir if missing
os.makedirs('models/disfluency', exist_ok=True)
disf = 'models/disfluency/best_model.pt'
if os.path.exists(disf) and os.path.getsize(disf) > 1_000_000:
    print(f'\u2705 Disfluency model: {os.path.getsize(disf)/1e6:.1f}MB')
else:
    print('\u26a0\ufe0f Disfluency model: missing (378MB — upload via Google Drive if available)')

# Check all models
print('\n--- Model Check ---')
for name, path in {
    'pose_landmarker': 'models/pose_landmarker_lite.task',
    'hand_landmarker': 'models/hand_landmarker.task',
    'face_landmarker': 'models/face_landmarker.task',
    'gesture_transformer': 'models/gesture_transformer/best_model.pt',
    'facial_emotion': 'models/facial_emotion/best_model.pt',
    'posture_mlp': 'models/posture_mlp/best_model.pt',
    'filler_verifier': 'models/filler_verifier/best_model.pt',
    'disfluency': 'models/disfluency/best_model.pt',
    'vocal_emotion': 'models/vocal_emotion/best_model.pt',
}.items():
    if os.path.exists(path):
        s = os.path.getsize(path)
        print(f'  {name}: \u2705 {s/1e6:.1f}MB' if s > 1000 else f'  {name}: \u26a0\ufe0f {s}B')
    else:
        print(f'  {name}: \u274c MISSING')

In [ ]:
#@title 3c. Download disfluency model from Google Drive
#@markdown Downloads best_model.pt directly — no Drive mounting needed.
import os

os.makedirs('/content/voice-analysis-pipeline/models/disfluency', exist_ok=True)
dst = '/content/voice-analysis-pipeline/models/disfluency/best_model.pt'

if os.path.exists(dst) and os.path.getsize(dst) > 100_000_000:
    print(f'\u2705 Disfluency model already present: {os.path.getsize(dst)/1e6:.1f}MB')
else:
    FILE_ID = '1zuBRwRG_a3kUsTZytLdl80ml29fuI9E9'
    !pip install -q gdown
    !gdown --id {FILE_ID} -O {dst}
    if os.path.exists(dst) and os.path.getsize(dst) > 100_000_000:
        print(f'\u2705 Disfluency model downloaded: {os.path.getsize(dst)/1e6:.1f}MB')
    else:
        print('\u274c Download failed. Make sure the file is shared as "Anyone with the link"')


In [ ]:
#@title 4. Setup ngrok tunnel
#@markdown Get a free auth token at https://dashboard.ngrok.com/signup
NGROK_AUTH_TOKEN = '' #@param {type:"string"}

from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print('\u2705 ngrok authenticated')
else:
    print('\u26a0\ufe0f No ngrok token — tunnel will work but may be rate-limited')
    print('   Get a free token at: https://dashboard.ngrok.com/signup')

In [ ]:
#@title 4b. Connect Claude Proxy (for LLM features)
#@markdown Paste the ngrok URL from your local Claude proxy here.
#@markdown Run `python scripts/claude_proxy.py --ngrok` on your local machine first.
CLAUDE_PROXY_URL = '' #@param {type:"string"}

import os
if CLAUDE_PROXY_URL:
    url = CLAUDE_PROXY_URL.strip().rstrip('/')
    os.environ['CLAUDE_PROXY_URL'] = url
    # Verify connection with ngrok header
    import requests
    try:
        r = requests.get(f'{url}/health',
                         headers={'ngrok-skip-browser-warning': 'true'},
                         timeout=5)
        if r.ok:
            print(f'\u2705 Claude proxy connected: {url}')
            print('   LLM features (tone, regime, argument, coaching) will use Claude via your Max subscription')
        else:
            print(f'\u26a0\ufe0f Proxy responded with status {r.status_code}')
    except Exception as e:
        print(f'\u26a0\ufe0f Could not reach proxy: {e}')
        print('   LLM features will use heuristic fallbacks (still works, lower quality)')
else:
    print('\u2139\ufe0f No Claude proxy URL — LLM features will use heuristic fallbacks')
    print('   To enable: run `python scripts/claude_proxy.py --ngrok` locally and paste the URL above')

In [ ]:
#@title 5. Pre-load Models (warm up GPU)
import sys
sys.path.insert(0, REPO_DIR)

print('Loading master pipeline...')
from src.master_pipeline import SpeechCoachPipeline
master = SpeechCoachPipeline()

print('Loading voice pipeline...')
try:
    from src.pipeline import VoiceAnalysisPipeline
    master._voice_pipeline = VoiceAnalysisPipeline()
    print('  \u2705 Voice pipeline ready')
except Exception as e:
    print(f'  \u26a0\ufe0f Voice pipeline: {e}')

print('Loading body pipeline...')
try:
    from src.body.pipeline import BodyAnalysisPipeline
    master._body_pipeline = BodyAnalysisPipeline()
    print('  \u2705 Body pipeline ready')
except Exception as e:
    print(f'  \u26a0\ufe0f Body pipeline: {e}')

print('Loading content pipeline...')
try:
    from src.content.pipeline import ContentAnalysisPipeline
    master._content_pipeline = ContentAnalysisPipeline()
    print('  \u2705 Content pipeline ready')
except Exception as e:
    print(f'  \u26a0\ufe0f Content pipeline: {e}')

# Inject the pre-loaded pipeline into the API server so it reuses these models
from src.api_server import set_pipeline
set_pipeline(master)

print('\n\u2705 All models loaded on GPU and injected into API server')

In [ ]:
#@title 6. Start GPU Backend Server
#@markdown This cell runs forever — the server stays up as long as Colab is connected.

import threading
import asyncio
import uvicorn
from pyngrok import ngrok

# Import the API server
from src.api_server import app

# Kill any existing server on port 8000
import subprocess
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
import time; time.sleep(1)

# Create ngrok tunnel
public_url = ngrok.connect(8000, 'http')
public_url_str = str(public_url).split('"')[1] if '"' in str(public_url) else str(public_url)

print('=' * 60)
print('  SPEECH COACH GPU BACKEND IS LIVE')
print('=' * 60)
print(f'')
print(f'  Public URL: {public_url_str}')
print(f'')
print(f'  Paste this URL in your webapp:')
print(f'  Settings -> COLAB_BACKEND_URL -> {public_url_str}')
print(f'')
print('=' * 60)

# Run uvicorn in its own thread with its own event loop
def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Keep cell alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('\nServer stopped')